# Semana 7 · Protección de datos personales en la práctica
**Gestión de Datos · IN1232C · 2026-2 · Cátedra del martes 15 de septiembre**

Durante seis semanas la pregunta fue **¿puedo hacerlo?**: cargar, limpiar, agrupar, graficar.
Hoy cambia a otra: **¿me está permitido hacerlo, y bajo qué condiciones?**

Trabajamos con la base de un centro de salud del Gran Concepción. Tiene nombre, RUT, comuna y
diagnóstico. En la S6 habríamos partido con un `head()`; hoy partimos preguntando qué de todo
esto se puede tocar, quién puede verlo y qué hay que hacer antes de compartirlo.

**Al terminar deberías poder:** clasificar las columnas de un dataset según su condición legal,
medir el riesgo de reidentificación, anonimizar de verdad una base y justificar cada decisión.

## Parte 0 · Carga y primer vistazo

In [1]:
import pandas as pd
import numpy as np
import hashlib, time

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

at = pd.read_csv("atenciones_centro_salud.csv", parse_dates=["fecha_nacimiento","fecha_atencion"])
print("Atenciones:", at.shape)
at.head(3)

Atenciones: (3200, 11)


,id_atencion,rut,nombre,fecha_nacimiento,sexo,comuna,fecha_atencion,especialidad,diagnostico,prevision,monto_total
0,AT-40089,12376776-1,Tomás Valenzuela Flores,1981-05-07,M,Concepción,2026-03-02,Traumatología,Tendinitis,FONASA B,95300
1,AT-40156,19548416-4,Tomás Castillo Fuentes,2016-12-06,M,San Pedro de la Paz,2026-03-02,Pediatría,Dermatitis atópica,Isapre,37500
2,AT-40157,12575058-0,Emilia Tapia Araya,1987-04-17,F,Chiguayante,2026-03-02,Medicina General,Lumbago,Isapre,30100


In [ ]:
# ¿Cuántas personas distintas hay detrás de esas atenciones?
print("Filas (atenciones)  :", len(at))
print("Personas distintas  :", at["rut"].nunique())
print("Período             :", at["fecha_atencion"].min().date(), "→", at["fecha_atencion"].max().date())
at.dtypes

Filas (atenciones)  : 3200
Personas distintas  : 1256
Período             : 2026-03-02 → 2026-08-31


id_atencion                 object
rut                         object
nombre                      object
fecha_nacimiento    datetime64[ns]
sexo                        object
comuna                      object
fecha_atencion      datetime64[ns]
especialidad                object
diagnostico                 object
prevision                   object
monto_total                  int64
dtype: object

## Parte 1 · Antes de tocar nada: clasificar las columnas

La ley no trata igual a todas las columnas. Hay tres categorías que conviene distinguir:

| Categoría | Qué es | Ejemplos aquí |
|---|---|---|
| **Identificador directo** | Por sí solo apunta a una persona | `rut`, `nombre` |
| **Cuasi-identificador** | Por sí solo no identifica, pero **combinado con otros, sí** | `fecha_nacimiento`, `sexo`, `comuna` |
| **Dato sensible** | Su filtración causa daño y la ley lo protege con un estándar más alto | `diagnostico` (salud) |

El resto —`especialidad`, `monto_total`, `prevision`— son datos del servicio, no de la persona.

La categoría que más se subestima es la del medio. Volvamos sobre ella en un minuto.

In [2]:
clasificacion = {
    "id_atencion":      "operacional",
    "rut":              "identificador directo",
    "nombre":           "identificador directo",
    "fecha_nacimiento": "cuasi-identificador",
    "sexo":             "cuasi-identificador",
    "comuna":           "cuasi-identificador",
    "fecha_atencion":   "cuasi-identificador",
    "especialidad":     "atributo del servicio",
    "diagnostico":      "DATO SENSIBLE (salud)",
    "prevision":        "atributo del servicio",
    "monto_total":      "atributo del servicio",
}
pd.DataFrame({"columna": clasificacion.keys(), "categoría": clasificacion.values()})

,columna,categoría
0,id_atencion,operacional
1,rut,identificador directo
2,nombre,identificador directo
3,fecha_nacimiento,cuasi-identificador
4,sexo,cuasi-identificador
5,comuna,cuasi-identificador
6,fecha_atencion,cuasi-identificador
7,especialidad,atributo del servicio
8,diagnostico,DATO SENSIBLE (salud)
9,prevision,atributo del servicio


## Parte 2 · La anonimización ingenua

Lo que casi todo el mundo hace la primera vez: borrar el nombre y el RUT, y dar la base por
anónima. Hagámoslo.

In [3]:
anon = at.drop(columns=["rut", "nombre"])
print("Ya no está el nombre ni el RUT. ¿Está anónima?")
anon.head(3)

Ya no está el nombre ni el RUT. ¿Está anónima?


,id_atencion,fecha_nacimiento,sexo,comuna,fecha_atencion,especialidad,diagnostico,prevision,monto_total
0,AT-40089,1981-05-07,M,Concepción,2026-03-02,Traumatología,Tendinitis,FONASA B,95300
1,AT-40156,2016-12-06,M,San Pedro de la Paz,2026-03-02,Pediatría,Dermatitis atópica,Isapre,37500
2,AT-40157,1987-04-17,F,Chiguayante,2026-03-02,Medicina General,Lumbago,Isapre,30100


Se ve anónima. **No lo está.**

Fijémonos en la terna `fecha_nacimiento` + `sexo` + `comuna`: ninguna de las tres identifica
a nadie por separado. Juntas son casi una huella digital.

In [4]:
llave = ["fecha_nacimiento", "sexo", "comuna"]

# ¿cuántas PERSONAS quedan solas en su combinación?
personas = at.drop_duplicates("rut")[["rut"] + llave]
tam_grupo = personas.groupby(llave)["rut"].transform("size")

unicas = (tam_grupo == 1).sum()
print(f"Personas en la base            : {len(personas)}")
print(f"Solas en su combinación        : {unicas}  ({unicas/len(personas)*100:.1f} %)")
print()
print("Es decir: para casi cualquier paciente, esas tres columnas juntas")
print("apuntan a una única persona en todo el archivo.")

Personas en la base            : 1256
Solas en su combinación        : 1250  (99.5 %)

Es decir: para casi cualquier paciente, esas tres columnas juntas
apuntan a una única persona en todo el archivo.


## Parte 3 · El ataque: reidentificar con una fuente externa

Saber que alguien es *único* todavía no es saber *quién* es. Para eso hace falta una segunda
fuente que tenga la misma terna **y además el nombre**.

Ese tipo de listas existe y es fácil de conseguir: un padrón de socios de un club, un listado
de una junta de vecinos, una base de un concurso. Aquí usamos `padron_vecinal.csv`.

In [5]:
padron = pd.read_csv("padron_vecinal.csv", parse_dates=["fecha_nacimiento"])
print("Padrón externo:", padron.shape)
padron.head(3)

Padrón externo: (868, 6)


,n_socio,nombre,fecha_nacimiento,sexo,comuna,correo
0,1118,Catalina Henríquez Valenzuela,1952-08-02,F,Talcahuano,chenríquez62@correo.cl
1,1592,Antonia Díaz Rodríguez,1990-11-03,F,San Pedro de la Paz,adíaz12@correo.cl
2,1419,Tomás Valenzuela Flores,1981-05-07,M,Concepción,tvalenzuela88@correo.cl


In [6]:
# El ataque es un merge. Nada más que eso.
# Un cruce es concluyente cuando la terna apunta a una sola persona en CADA archivo.
pad_unico = padron.groupby(llave)["n_socio"].transform("size") == 1
padron_u  = padron[pad_unico]

expuestas = anon.merge(padron_u, on=llave, how="inner", suffixes=("", "_padron"))

print(f"Personas identificadas  : {expuestas['nombre'].nunique()}")
print(f"Atenciones expuestas    : {len(expuestas)} de {len(anon)}")
print(f"                          ({len(expuestas)/len(anon)*100:.0f} % de la base, con su diagnóstico)")
expuestas[["nombre", "correo", "comuna", "especialidad", "diagnostico", "fecha_atencion"]].head(8)

Personas identificadas  : 761
Atenciones expuestas    : 1935 de 3200
                          (60 % de la base, con su diagnóstico)


,nombre,correo,comuna,especialidad,diagnostico,fecha_atencion
0,Tomás Valenzuela Flores,tvalenzuela88@correo.cl,Concepción,Traumatología,Tendinitis,2026-03-02
1,Tomás Castillo Fuentes,tcastillo61@correo.cl,San Pedro de la Paz,Pediatría,Dermatitis atópica,2026-03-02
2,Emilia Tapia Araya,etapia72@correo.cl,Chiguayante,Medicina General,Lumbago,2026-03-02
3,Vicente Muñoz Vergara,vmunoz52@correo.cl,Talcahuano,Medicina General,Resfrío común,2026-03-02
4,Emilia Gutiérrez Salazar,egutiérrez18@correo.cl,Concepción,Ginecología,Síndrome de ovario poliquístico,2026-03-02
5,Florencia Martínez Henríquez,fmartínez19@correo.cl,Talcahuano,Pediatría,Dermatitis atópica,2026-03-02
6,Cristóbal Henríquez Díaz,chenríquez44@correo.cl,Concepción,Medicina General,Lumbago,2026-03-02
7,Josefa Reyes Díaz,jreyes26@correo.cl,Concepción,Ginecología,Control preventivo,2026-03-02


Eso es una filtración de datos de salud, y se produjo con un `merge` de una línea.

El diagnóstico psiquiátrico de una persona identificable quedó a la vista. No hubo hackeo, no
se vulneró ninguna contraseña: alguien publicó una base "anonimizada" y otro la cruzó con una
lista que estaba disponible.

**La lección:** anonimizar no es borrar el nombre. Es garantizar que nadie quede solo en su
combinación de atributos.

## Parte 4 · Medir el riesgo: k-anonimato

Una base cumple **k-anonimato** cuando cada combinación de cuasi-identificadores se repite
al menos `k` veces. Con `k = 1` hay gente sola: reidentificable. Con `k = 5`, cualquier
combinación corresponde al menos a cinco personas, y ya no se puede señalar a una.

`k` es el tamaño **del grupo más pequeño**.

In [7]:
def k_anonimato(df, cols):
    """Devuelve el k de la base y la distribución de tamaños de grupo.

    observed=True es importante: con columnas categóricas, pandas inventa por
    defecto todas las combinaciones posibles, incluidas las que no existen, y
    esos grupos vacíos de tamaño 0 arruinan el mínimo."""
    tam = df.groupby(cols, observed=True).size()
    return tam.min(), tam

k, tam = k_anonimato(anon, llave)
print(f"k de la base 'anonimizada' = {k}")
print()
print("Distribución del tamaño de los grupos:")
print(tam.value_counts().sort_index().head(8).to_string())

k de la base 'anonimizada' = 1

Distribución del tamaño de los grupos:
1     352
2     328
3     290
4     155
5      78
6      41
7       8
10      1


## Parte 5 · Anonimizar de verdad

Dos herramientas, y ninguna es gratis:

- **Generalizar** — bajar la precisión: la fecha de nacimiento pasa a tramo de edad, la comuna
  a provincia, la fecha exacta a mes.
- **Suprimir** — eliminar las filas que aun así siguen solas.

Cada paso protege más y sirve menos. La pregunta profesional no es "¿cómo anonimizo?", sino
**"¿cuánta utilidad estoy dispuesto a perder?"**.

In [ ]:
hoy = pd.Timestamp("2026-09-15")

seguro = at.drop(columns=["rut", "nombre"]).copy()

# 1 · fecha de nacimiento → tramo de edad de 10 años
edad = ((hoy - seguro["fecha_nacimiento"]).dt.days // 365).astype(int)
seguro["tramo_edad"] = pd.cut(edad, bins=[0,17,29,44,59,74,120],
                              labels=["0-17","18-29","30-44","45-59","60-74","75+"])

# 2 · comuna → provincia
provincia = {
    "Concepción":"Concepción","Talcahuano":"Concepción","San Pedro de la Paz":"Concepción",
    "Chiguayante":"Concepción","Hualpén":"Concepción","Penco":"Concepción",
    "Tomé":"Concepción","Coronel":"Concepción","Lota":"Concepción",
}
seguro["provincia"] = seguro["comuna"].map(provincia)

# 3 · fecha de atención → mes
seguro["mes_atencion"] = seguro["fecha_atencion"].dt.to_period("M").astype(str)

seguro = seguro.drop(columns=["fecha_nacimiento", "comuna", "fecha_atencion"])
llave2 = ["tramo_edad", "sexo", "provincia", "mes_atencion"]
seguro.head(3)

,id_atencion,sexo,especialidad,diagnostico,prevision,monto_total,tramo_edad,provincia,mes_atencion
0,AT-40089,M,Traumatología,Tendinitis,FONASA B,95300,45-59,Concepción,2026-03
1,AT-40156,M,Pediatría,Dermatitis atópica,Isapre,37500,0-17,Concepción,2026-03
2,AT-40157,F,Medicina General,Lumbago,Isapre,30100,30-44,Concepción,2026-03


In [ ]:
k2, tam2 = k_anonimato(seguro, llave2)
print(f"k después de generalizar = {k2}")
print()
print("Grupos que todavía quedan bajo 5:")
chicos = tam2[tam2 < 5]
print(f"  {len(chicos)} grupos, {chicos.sum()} filas en total")

k después de generalizar = 2

Grupos que todavía quedan bajo 5:
  1 grupos, 2 filas en total


In [ ]:
# 4 · suprimir lo que sigue quedando solo
OBJETIVO = 5
tam_fila = seguro.groupby(llave2, observed=True)["id_atencion"].transform("size")
publicable = seguro[tam_fila >= OBJETIVO].copy()

k3, _ = k_anonimato(publicable, llave2)
perdido = len(seguro) - len(publicable)
print(f"k final               : {k3}")
print(f"Filas publicables     : {len(publicable)} de {len(seguro)}")
print(f"Filas suprimidas      : {perdido} ({perdido/len(seguro)*100:.1f} % de la base)")

k final               : 5
Filas publicables     : 3180 de 3200
Filas suprimidas      : 20 (0.6 % de la base)


In [ ]:
# ¿Sirve todavía para algo? Comparemos un indicador antes y después.
antes   = at.groupby("especialidad")["monto_total"].mean().round(0)
despues = publicable.groupby("especialidad")["monto_total"].mean().round(0)
comp = pd.DataFrame({"base original": antes, "base publicable": despues})
comp["diferencia %"] = ((comp["base publicable"]/comp["base original"] - 1)*100).round(1)
comp

,base original,base publicable,diferencia %
especialidad,,,
Cardiología,55474.0,55477.0,0.0
Dermatología,54110.0,54110.0,0.0
Ginecología,53461.0,53461.0,0.0
Laboratorio,51690.0,51522.0,-0.3
Medicina General,52558.0,52549.0,-0.0
Pediatría,51223.0,51171.0,-0.1
Psiquiatría,55950.0,55926.0,-0.0
Traumatología,53383.0,53309.0,-0.1


Ese último cuadro es el argumento que hay que saber dar en una reunión: **la base anonimizada
conserva el indicador** que al área de gestión le interesa. Se perdió detalle individual, no se
perdió la respuesta.

## Parte 6 · Seudonimizar: el hash y su trampa

A veces no se puede borrar el identificador, porque hay que seguir a la misma persona entre
tablas. Ahí se **seudonimiza**: se reemplaza el RUT por un código estable.

La primera idea de todo el mundo es aplicar un hash. Veamos si aguanta.

In [ ]:
def hash_simple(rut):
    return hashlib.sha256(str(rut).encode()).hexdigest()[:16]

demo = at[["rut"]].drop_duplicates().head(5).copy()
demo["seudonimo"] = demo["rut"].map(hash_simple)
demo

,rut,seudonimo
0,12376776-1,cfca6f10fcb06884
1,19548416-4,441ba9628185b979
2,12575058-0,01a1ccae6bbc3f18
3,15453849-0,80139b3c9c1ec753
4,12536058-8,aa24e743e4632f25


Se ve irreversible. Y no lo es, por una razón que no tiene que ver con el algoritmo sino
con el **tamaño del espacio de búsqueda**: los RUT chilenos son un rango acotado y conocido.
Se pueden generar todos.

In [ ]:
objetivo = demo["seudonimo"].iloc[0]          # un seudónimo cualquiera
rut_real = demo["rut"].iloc[0]
inicio = int(rut_real.split("-")[0]) - 40_000   # buscamos en una ventana, para no esperar

t0 = time.time()
encontrado = None
probados = 0
for num in range(inicio, inicio + 120_000):
    for d in "0123456789K":
        probados += 1
        if hash_simple(f"{num}-{d}") == objetivo:
            encontrado = f"{num}-{d}"
            break
    if encontrado:
        break
t = time.time() - t0

print(f"Seudónimo  : {objetivo}")
print(f"RUT hallado: {encontrado}")
print(f"Probados   : {probados:,} combinaciones en {t:.2f} s")
print()
vel = probados / t
print(f"A esta velocidad ({vel:,.0f}/s), recorrer los ~30 millones de RUT posibles")
print(f"toma del orden de {30_000_000/vel:.0f} segundos. Un hash sin sal no protege nada.")

Seudónimo  : cfca6f10fcb06884
RUT hallado: 12376776-1
Probados   : 440,002 combinaciones en 0.37 s

A esta velocidad (1,179,169/s), recorrer los ~30 millones de RUT posibles
toma del orden de 25 segundos. Un hash sin sal no protege nada.


In [ ]:
# La forma correcta: hash con sal secreta (HMAC). La sal vive aparte de los datos.
SAL = "clave-institucional-que-no-viaja-con-el-archivo"

def seudonimo(rut, sal=SAL):
    import hmac
    return hmac.new(sal.encode(), str(rut).encode(), hashlib.sha256).hexdigest()[:16]

demo["seudonimo_con_sal"] = demo["rut"].map(seudonimo)
demo[["rut", "seudonimo", "seudonimo_con_sal"]]

,rut,seudonimo,seudonimo_con_sal
0,12376776-1,cfca6f10fcb06884,7bbf79d786784c8f
1,19548416-4,441ba9628185b979,a3ab1101fa4a4cd3
2,12575058-0,01a1ccae6bbc3f18,1d7a6884bdfb200d
3,15453849-0,80139b3c9c1ec753,47c668cc21003f6b
4,12536058-8,aa24e743e4632f25,335dfcb77d211c7a


Ahora el atacante ya no puede generar candidatos: sin la sal, probar los 30 millones de RUT
no sirve de nada.

**Dos advertencias.** Primero, seudonimizar **no es** anonimizar: si existe la sal, existe la
posibilidad de revertir, y por eso la base sigue siendo de datos personales para efectos
legales. Segundo, el seudónimo es estable, así que todas las atenciones de una persona siguen
enlazadas — que es justamente lo que se quería, pero también lo que permite perfilarla.

## Parte 7 · Política de acceso: quién ve qué

El último control no está en el dato sino en el **acceso**. El principio es el de **mínimo
privilegio**: cada rol ve lo que necesita para su función, y nada más.

In [ ]:
politica = pd.DataFrame(
    [
        ["Médico tratante",   "Sí", "Sí", "Solo sus pacientes",     "Sí"],
        ["Personal de SOME",  "Sí", "No", "Agenda del día",         "No"],
        ["Analista de datos", "No", "Sí", "Toda la base, anónima",  "No"],
        ["Finanzas",          "No", "No", "Montos agregados",       "No"],
        ["Investigación",     "No", "Sí", "Base con k ≥ 5",         "No"],
    ],
    columns=["Rol", "Ve identidad", "Ve diagnóstico", "Alcance", "Puede modificar"],
)
politica

,Rol,Ve identidad,Ve diagnóstico,Alcance,Puede modificar
0,Médico tratante,Sí,Sí,Solo sus pacientes,Sí
1,Personal de SOME,Sí,No,Agenda del día,No
2,Analista de datos,No,Sí,"Toda la base, anónima",No
3,Finanzas,No,No,Montos agregados,No
4,Investigación,No,Sí,Base con k ≥ 5,No


In [ ]:
def vista(df, rol):
    """Devuelve la porción de la base que corresponde a un rol."""
    if rol == "analista":
        return df.drop(columns=["rut", "nombre"])
    if rol == "finanzas":
        return (df.groupby(["especialidad", "prevision"], as_index=False)["monto_total"]
                  .agg(["count", "sum"]))
    if rol == "some":
        return df[["id_atencion", "nombre", "fecha_atencion", "especialidad"]]
    raise ValueError(f"Rol sin política definida: {rol}")

print("Vista de FINANZAS — ni identidad ni diagnóstico, solo el agregado:")
vista(at, "finanzas").head(6)

Vista de FINANZAS — ni identidad ni diagnóstico, solo el agregado:


,especialidad,prevision,count,sum
0,Cardiología,FONASA A,53,2883400
1,Cardiología,FONASA B,68,3695600
2,Cardiología,FONASA C,63,3427300
3,Cardiología,FONASA D,44,2442100
4,Cardiología,Isapre,83,4796300
5,Cardiología,Particular,6,340500


In [ ]:
print("Vista de SOME — necesita el nombre para llamar al paciente, pero no el diagnóstico:")
vista(at, "some").head(4)

Vista de SOME — necesita el nombre para llamar al paciente, pero no el diagnóstico:


,id_atencion,nombre,fecha_atencion,especialidad
0,AT-40089,Tomás Valenzuela Flores,2026-03-02,Traumatología
1,AT-40156,Tomás Castillo Fuentes,2026-03-02,Pediatría
2,AT-40157,Emilia Tapia Araya,2026-03-02,Medicina General
3,AT-40316,Antonella Castillo Hernández,2026-03-02,Pediatría


Fíjate en la última: el personal de admisión **sí** ve el nombre, porque sin él no puede
hacer su trabajo. Mínimo privilegio no significa esconderlo todo, significa entregar lo justo.

---
# Desafíos

Para resolver después de la clase. No se entregan, pero el **Test 2** pregunta sobre esto.

**1.** La base `publicable` llegó a `k = 5`. Repite el proceso apuntando a `k = 10`.
¿Cuántas filas más hay que suprimir? ¿Sigue sirviendo el indicador de la Parte 5?

**2.** El `diagnostico` es dato sensible. Un grupo puede tener `k = 5` y aun así filtrar: si
las cinco personas del grupo tienen el mismo diagnóstico, saber que alguien está en ese grupo
revela su diagnóstico. Busca en `publicable` los grupos donde eso ocurra.
*(Pista: es lo que se llama diversidad-ℓ. Agrupa por `llave2` y cuenta diagnósticos distintos.)*

**3.** Un investigador pide la base para estudiar la demanda por especialidad según edad.
¿Qué columnas le entregarías y con qué generalización? Justifica qué pierde y qué conserva.

**4.** El correo del padrón tiene la forma `inicial+apellido+número`. ¿Es dato personal?
¿Cambiaría tu respuesta si el número fuera aleatorio y no hubiera padrón?

**5.** Escribe la función `vista(df, "investigacion")` que falta, coherente con la tabla de
política de la Parte 7.